### Building a RAG System with LangChain and ChromaDB
Introduction
Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:
<ul>
<li>LangChain: A framework for developing applications powered by language models </li>
<li>ChromaDB: An open-source vector database for storing and retrieving embeddings </li>
<li>OpenAI: For embeddings and language model (you can substitute with other providers like HuggingFace or Grok) </li>
</ul>

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
file_path = "data/text_files/doc_1.txt"
print(os.path.exists(file_path))  # Should print True
# Load the document
loader=TextLoader("data/text_files/doc_1.txt", encoding="utf-8")
documents = loader.load()
# Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

texts = text_splitter.split_documents(documents)
# Create the embedding model
embedding_model = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
# Create the Chroma vector store
vectorstore = Chroma.from_documents(texts, embedding_model, collection_name='shakespeare')

# Example query
query = "What is the main theme of Hamlet?"
# Get the relevant documents
results = vectorstore.similarity_search(query, k=3)
for i, result in enumerate(results):
    print(f"Result {i+1}:")
    print(result.page_content)
    print("\n---\n")

True


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Result 1:
Machine Learning Fundamentals

    Machine Learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are three main
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised
    learning find patterns in unlabeled data. Reinforcement learning learns through
    interactions with an environment using rewards and penalties.

---



RAG (Retrieval-Augmented Generation) Architecture:
1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduce hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge

In [6]:
# langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma

# Utility imports
import numpy as np
from typing import List

# Create Sample Data (list of documents)
sample_docs = [
    """
    Machine Learning Fundamentals

    Machine Learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are three main
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised
    learning find patterns in unlabeled data. Reinforcement learning learns through
    interactions with an environment using rewards and penalties.
    """,
    """
    Deep Learning and Neural Networks
    
    Deep Learning is a subset of machine learning based on aritificial neural networks.
    These networks are inspired by the human brain and consist of layers of interconnected
    nodes. Deep learning has revolutionlized fields like computer vision, natural language
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers
    excel at handling sequential data like text and time series.
    """,
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language.
    Key tasks in NLP include text classification, sentiment analysis, named entity recognition, 
    machine translation, and question answering. Modern NLP heavily relies on transformer
    architectures like BERT, GPT and T5, which have significantly improved the ability of machines
    to understand and generate human language. Applications of NLP range from chatbots and virtual 
    assistants to automated content generation and language translation services. These models use
    attention mechanisms to capture the context and relationships between words, enabling more accurate
    language understanding and generation.
    """
]
sample_docs

# Save sample documents to text files

os.makedirs("data/text_files", exist_ok=True)
for i, doc in enumerate(sample_docs):
    with open(f"data/text_files/doc_{i+1}.txt", "w", encoding="utf-8") as f:
        f.write(doc.strip())

print(f"Sample documents saved to: data/text_files/")

# Load the documents using TextLoader
loader = DirectoryLoader("data/text_files", 
                         glob="*.txt", 
                         loader_cls=TextLoader,
                         loader_kwargs={"encoding": "utf-8"}
        )
documents = loader.load()
print(f"Loaded {len(documents)} document(s).")
print(f"\nFirst document content:\n{documents[0].page_content[:500]}...")  # Print the first 500 characters of the first document

# Split the documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,
                                                chunk_overlap=50,
                                                length_function=len,
                                               #separators=["\n\n", "\n", ". ", " ", ""] # Define separators for splitting
                                                separators=[" "]
                                                )
chunks = text_splitter.split_documents(documents)
print(f"\nSplit into {len(chunks)} chunks from {len(documents)} document(s).")
print(f"\nFirst chunk content:\n{chunks[0].page_content[:150]}...")  # Print the first 500 characters of the first chunk 
print(f"\nFirst chunk metadata:\n{chunks[0].metadata}")

chunks

# To set the environment variable in the current session (optional, since load_dotenv already does this) or current workspace
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# Create the embedding model
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1536)

# Initialize the chromadb vector store and store the chunks in vector representation

# Create a ChromaDB vector store
persist_directory="./chroma_db"

# Initialize Chromadb with OpenAI embeddings
# embeddings will be created from chunks, will store in persist_directory
# name of vector store will be rag_collection
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=persist_directory,
    collection_name="rag_collection"
)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to: {persist_directory}")

Sample documents saved to: data/text_files/
Loaded 3 document(s).

First document content:
Machine Learning Fundamentals

    Machine Learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are three main
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised
    learning find patterns in unlabeled data. Reinforcement learning learns through
    interactions with a...

Split into 6 chunks from 3 document(s).

First chunk content:
Machine Learning Fundamentals

    Machine Learning is a subset of artificial intelligence that enables systems to learn
    and improve from experien...

First chunk metadata:
{'source': 'data\\text_files\\doc_1.txt'}
Vector store created with 12 vectors
Persisted to: ./chroma_db


### Text Similarity Search

In [7]:
# How to do Similarity Search with ChromaDB and OpenAI Embeddings
query="What are the types of machine learning?"
results = vectorstore.similarity_search(query, k=3)
results


[Document(metadata={'source': 'data\\text_files\\doc_1.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine Learning is a subset of artificial intelligence that enables systems to learn\n    and improve from experience without being explicitly programmed. There are three main\n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised\n    learning find patterns in unlabeled data. Reinforcement learning learns through\n    interactions with'),
 Document(metadata={'source': 'data\\text_files\\doc_1.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine Learning is a subset of artificial intelligence that enables systems to learn\n    and improve from experience without being explicitly programmed. There are three main\n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised lear

In [8]:
print(f"Top {len(results)} results for query: '{query}'\n")
for i, result in enumerate(results):
    print(f"Result {i+1}:")
    print(result.page_content)
    print("\n---\n")

Top 3 results for query: 'What are the types of machine learning?'

Result 1:
Machine Learning Fundamentals

    Machine Learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are three main
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised
    learning find patterns in unlabeled data. Reinforcement learning learns through
    interactions with

---

Result 2:
Machine Learning Fundamentals

    Machine Learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are three main
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised
    learning f

In [9]:
query="What is NLP?"
results = vectorstore.similarity_search(query, k=3)
results

[Document(metadata={'source': 'data\\text_files\\doc_3.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language.\n    Key tasks in NLP include text classification, sentiment analysis, named entity recognition, \n    machine translation, and question answering. Modern NLP heavily relies on transformer\n    architectures like BERT, GPT and T5, which have significantly improved the ability of machines\n    to understand and generate human language. Applications of NLP range from'),
 Document(metadata={'source': 'data\\text_files\\doc_3.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language.\n    Key tasks in NLP include text classification, sentiment analysis, named entity recognition, \n    machine translation, and question answering. Modern NLP heavily relies on transformer\n    architectures like 

In [10]:
query="What is Deep Learning?"
results = vectorstore.similarity_search(query, k=3)
results

[Document(metadata={'source': 'data\\text_files\\doc_2.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep Learning is a subset of machine learning based on aritificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionlized fields like computer vision, natural language\n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly\n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers\n    excel'),
 Document(metadata={'source': 'data\\text_files\\doc_2.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep Learning is a subset of machine learning based on aritificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionlized fields like computer vision, natural language\n    processing, and speech 

In [11]:
print(f"Query: '{query}'\n")
print(f"Top {len(results)} results for query: '{query}'\n")
for i, result in enumerate(results):
    print(f"Chunk {i+1}:")
    print(result.page_content[:100] + "...")
    print(f"Source: {result.metadata.get('source', 'Unknown')}")
    print("\n---\n")    

Query: 'What is Deep Learning?'

Top 3 results for query: 'What is Deep Learning?'

Chunk 1:
Deep Learning and Neural Networks

    Deep Learning is a subset of machine learning based on aritif...
Source: data\text_files\doc_2.txt

---

Chunk 2:
Deep Learning and Neural Networks

    Deep Learning is a subset of machine learning based on aritif...
Source: data\text_files\doc_2.txt

---

Chunk 3:
Machine Learning Fundamentals

    Machine Learning is a subset of artificial intelligence that enab...
Source: data\text_files\doc_1.txt

---



### Similarity Search with Score

In [12]:
results_scores=vectorstore.similarity_search_with_score(query, k=3)
results_scores

[(Document(metadata={'source': 'data\\text_files\\doc_2.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep Learning is a subset of machine learning based on aritificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionlized fields like computer vision, natural language\n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly\n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers\n    excel'),
  0.7202814817428589),
 (Document(metadata={'source': 'data\\text_files\\doc_2.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep Learning is a subset of machine learning based on aritificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionlized fields like computer vision, natural language\n  

ChromaDB by default uses L2 distance (Euclidean distance) for Similarity Search
- Lowest score = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typically range: 0 to 2 (but can be higher)

### Initialize LLM, RAG Chain, Prompt Template, Query the RAG system

In [13]:
# One way to initialize ChatOpenAI model
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", 
                 temperature=0.2,
                 max_tokens=500)

text_response = llm.invoke("What is Large Lanaguage Models?")
text_response

AIMessage(content="Large Language Models (LLMs) are a type of artificial intelligence model that is trained on vast amounts of text data to understand and generate human language. These models are typically based on deep learning techniques, such as transformers, and are capable of performing a wide range of natural language processing tasks, such as text generation, translation, summarization, and question-answering.\n\nSome well-known examples of LLMs include OpenAI's GPT (Generative Pre-trained Transformer) models and Google's BERT (Bidirectional Encoder Representations from Transformers) model. These models have been used in a variety of applications, including chatbots, language translation, content generation, and more.\n\nLLMs have the potential to revolutionize the field of natural language processing and enable more advanced and human-like interactions between humans and machines. However, there are also concerns about the ethical implications of using such powerful language m

In [14]:
# Another way to initialize ChatOpenAI model
from langchain.chat_models.base import init_chat_model
chat_model = init_chat_model(
    model="gpt-3.5-turbo",
    temperature=0.2,
    max_tokens=500
)

llm=init_chat_model("openai:gpt-3.5-turbo")
#llm=init_chat_model("groq:modelname")
llm
llm.invoke("What is AI?")
                    

AIMessage(content='AI, or artificial intelligence, refers to the development of computer systems that can perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation. AI technologies include machine learning, deep learning, natural language processing, and computer vision, among others. AI is used in a wide range of applications, including virtual assistants, autonomous vehicles, healthcare diagnostics, and fraud detection.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 11, 'total_tokens': 93, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DLsEPRbXJb08UWlcC9bmit8VhAJG3', 

### Modern RAG Chain - Using LCEL (LangChain Expression Lanaguage)

In [15]:
from langchain_core.runnables import Runnable
retriever = vectorstore.as_retriever()
retriever
retriever.invoke("What is Deep Learning?")

[Document(metadata={'source': 'data\\text_files\\doc_2.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep Learning is a subset of machine learning based on aritificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionlized fields like computer vision, natural language\n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly\n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers\n    excel'),
 Document(metadata={'source': 'data\\text_files\\doc_2.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep Learning is a subset of machine learning based on aritificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionlized fields like computer vision, natural language\n    processing, and speech 

In [16]:
# -------------------------
# Imports
# -------------------------
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableMap, RunnablePassthrough
from langchain_openai import ChatOpenAI

# -------------------------
# Step 1: LLM
# -------------------------
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# -------------------------
# Step 2: Retriever
# -------------------------
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# -------------------------
# Step 3: Prompt
# -------------------------
system_prompt = """You are an assistant for question-answering tasks.
Use the retrieved context to answer the question.
If you don't know the answer, say you don't know.
Keep the answer concise (max 3 sentences).

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{query}")
    ]
)

# -------------------------
# Step 4: Format documents
# -------------------------
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# -------------------------
# Step 5: Build RAG pipeline
# -------------------------
rag_chain = (
    RunnableMap({
        "context": retriever | format_docs,   # retrieve + format
        "query": RunnablePassthrough()        # pass question as-is
    })
    | prompt
    | llm
)
rag_chain


{
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001CC28902410>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  query: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'query'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the retrieved context to answer the question.\nIf you don't know the answer, say you don't know.\nKeep the answer concise (max 3 sentences).\n\nContext:\n{context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, '

This chain:
- Takes retrieved documents
- "Stuffs" them into the prompt's {context} placeholder
- Sends the complete prompt to the LLM
- Returns the LLM's response

In [17]:
# -------------------------
# Step 6: Run
# -------------------------
question = "What is machine learning?"

response = rag_chain.invoke(question)

print("Question:", question)
print("Answer:", response.content)

Question: What is machine learning?
Answer: Machine learning is a subset of artificial intelligence that allows systems to learn and improve from experience without being explicitly programmed. It involves three main types: supervised learning, unsupervised learning, and reinforcement learning.


In [18]:
# -------------------------
# Imports
# -------------------------
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document

# -------------------------
# Step 1: LLM
# -------------------------
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# -------------------------
# Step 2: Retriever
# -------------------------
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Wrap retriever in Runnable (IMPORTANT FIX)
retriever_runnable = RunnableLambda(lambda query: retriever.invoke(query))

# -------------------------
# Step 3: Prompt
# -------------------------
system_prompt = """You are an assistant for question-answering tasks.
Use the retrieved context to answer the question.
If you don't know the answer, say you don't know.
Keep the answer concise (max 3 sentences).

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{query}")
    ]
)

# -------------------------
# Step 4: Format documents
# -------------------------
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# -------------------------
# Step 5: Build RAG chain
# -------------------------
rag_chain = (
    {
        "context": retriever_runnable | RunnableLambda(format_docs),
        "query": lambda x: x
    }
    | prompt
    | llm
)

# -------------------------
# Step 6: Run query
# -------------------------
question = "What is machine learning?"

response = rag_chain.invoke(question)

print("Question:", question)
print("Answer:", response)

Question: What is machine learning?
Answer: content='Machine learning is a subset of artificial intelligence that allows systems to learn and improve from experience without being explicitly programmed. It involves three main types: supervised learning, unsupervised learning, and reinforcement learning.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 338, 'total_tokens': 378, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DLsESpvRwZVmlf2V1TROhQwPd3Ust', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d10f1-1b22-7581-84cf-e29b13625f6b-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 338, 'output_tokens

In [19]:
response.content

'Machine learning is a subset of artificial intelligence that allows systems to learn and improve from experience without being explicitly programmed. It involves three main types: supervised learning, unsupervised learning, and reinforcement learning.'

In [20]:
question = "What is Deep learning?"

response = rag_chain.invoke(question)

print("Question:", question)
print("Answer:", response.content)

Question: What is Deep learning?
Answer: Deep learning is a subset of machine learning that is based on artificial neural networks inspired by the human brain. It consists of interconnected layers of nodes and has revolutionized fields like computer vision, natural language processing, and speech recognition.


In [21]:
def query_rag_modern(question):
    print(f"Question: {question}\n")
    print("-" * 50)

    # Step 1: Retrieve documents
    docs = retriever_runnable.invoke(question)  # list of Document objects

    # Step 2: Format documents into context string
    context_str = format_docs(docs)

    # Step 3: Print retrieved context
    print("\nRetrieved Context:\n")
    for i, doc in enumerate(docs, 1):
        print(f"Doc {i}: {doc.page_content}\n")

    # Step 4: Prepare prompt
    prompt_text = prompt.format(context=context_str, query=question)

    # Step 5: Invoke LLM
    answer = llm.invoke(prompt_text)

    print(f"Answer: {answer.content}\n")
    print("-" * 50)
    return {"answer": answer, "context": docs}

# Test queries
test_questions = [
    "What are the types of machine learning?",
    "What is NLP?",
    "What is Deep Learning?"
]

for q in test_questions:
    result = query_rag_modern(q)
    print("\n==============================\n")
    print(f"Question: {q}\nAnswer: {result['answer'].content}\n")
    print("\n==============================\n")


Question: What are the types of machine learning?

--------------------------------------------------

Retrieved Context:

Doc 1: Machine Learning Fundamentals

    Machine Learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are three main
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised
    learning find patterns in unlabeled data. Reinforcement learning learns through
    interactions with

Doc 2: Machine Learning Fundamentals

    Machine Learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are three main
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to tra

### Create RAG Chain Alternative using RunnableParallel

In [22]:
# Even more flexible approach using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question.
    If you don't know the answer based on the context, say you don't know.
    Provide specific details from the context to support your answer.
                                                 
    Context: {context}
                                                 
    Question: {query}
    
    Answer:""")

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Format the output documents for the retriever in specific format for the prompt
def format_docs(docs: List[Document]) -> str:
    formatted = "\n\n".join(f"Document {i+1}:\n{doc.page_content}" for i, doc in enumerate(docs))
    return formatted

def debug_and_format(docs):
    # Print retrieved docs
    print("===== Retrieved Documents inside chain =====")
    for i, doc in enumerate(docs, 1):
        print(f"Doc {i}: {doc.page_content}\n")
    print("===========================================")
    # Return formatted string for LLM
    return format_docs(docs)

# Build a more flexible RAG chain using LCEL
rag_chain_lcel = (
    RunnableParallel({
        #"context": retriever | RunnableLambda(format_docs),  # Retrieve and format context
        "context": retriever | debug_and_format,  # Retrieve and format context
        "query": RunnablePassthrough()  # Pass query as-is
    })
    | custom_prompt  # Use custom prompt
    | llm  # Generate answer with LLM
    | StrOutputParser()  # Parse output as string
)

# Test the LCEL RAG chain
for q in test_questions:    
    print(f"Question: {q}\n")
    answer = rag_chain_lcel.invoke(q)
    print(f"Answer: {answer}\n")
    print("-" * 50)

Question: What are the types of machine learning?

===== Retrieved Documents inside chain =====
Doc 1: Machine Learning Fundamentals

    Machine Learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are three main
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised
    learning find patterns in unlabeled data. Reinforcement learning learns through
    interactions with

Doc 2: Machine Learning Fundamentals

    Machine Learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are three main
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervis

### Add New Documents to Existing Vector Store

In [23]:
# This is chromadb that we will be using it here
vectorstore

In [24]:
# Add new documents to the existing ChromaDB vector store
new_document = """
Reinforcement Learning (RL)

Reinforcement Learning (RL) is a type of machine learning where an agent learns to make decisions by
interacting with an environment. The agent receives feedback in the form of rewards or penalties based on its actions, 
and its goal is to learn a policy that maximizes cumulative rewards over time. 

Key Concepts in RL: states, actions, rewards, policy, value function, and exploration vs. exploitation.
Popular RL algorithms in RL include Q-learning, Deep Q-Networks (DQN), and Proximal Policy Optimization (PPO).

RL has been successfully applied in various domains, including robotics, game playing (e.g., AlphaGo), and autonomous systems.
"""
new_doc = Document(page_content=new_document, metadata={"source": "manual_addition", "topic": "Reinforcement Learning"})
new_doc


Document(metadata={'source': 'manual_addition', 'topic': 'Reinforcement Learning'}, page_content='\nReinforcement Learning (RL)\n\nReinforcement Learning (RL) is a type of machine learning where an agent learns to make decisions by\ninteracting with an environment. The agent receives feedback in the form of rewards or penalties based on its actions, \nand its goal is to learn a policy that maximizes cumulative rewards over time. \n\nKey Concepts in RL: states, actions, rewards, policy, value function, and exploration vs. exploitation.\nPopular RL algorithms in RL include Q-learning, Deep Q-Networks (DQN), and Proximal Policy Optimization (PPO).\n\nRL has been successfully applied in various domains, including robotics, game playing (e.g., AlphaGo), and autonomous systems.\n')

In [25]:
chunks

[Document(metadata={'source': 'data\\text_files\\doc_1.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine Learning is a subset of artificial intelligence that enables systems to learn\n    and improve from experience without being explicitly programmed. There are three main\n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised\n    learning find patterns in unlabeled data. Reinforcement learning learns through\n    interactions with'),
 Document(metadata={'source': 'data\\text_files\\doc_1.txt'}, page_content='learning learns through\n    interactions with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\text_files\\doc_2.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep Learning is a subset of machine learning based on aritificial neural networks.\n    These networks are inspired by the huma

In [26]:
print(f"After adding new document, vector store now has {vectorstore._collection.count()} vectors.")

After adding new document, vector store now has 12 vectors.


In [27]:
# Split the document into chunks
new_chunks = text_splitter.split_documents([new_doc])
new_chunks

[Document(metadata={'source': 'manual_addition', 'topic': 'Reinforcement Learning'}, page_content='Reinforcement Learning (RL)\n\nReinforcement Learning (RL) is a type of machine learning where an agent learns to make decisions by\ninteracting with an environment. The agent receives feedback in the form of rewards or penalties based on its actions, \nand its goal is to learn a policy that maximizes cumulative rewards over time. \n\nKey Concepts in RL: states, actions, rewards, policy, value function, and exploration vs. exploitation.\nPopular RL algorithms in RL include Q-learning, Deep Q-Networks'),
 Document(metadata={'source': 'manual_addition', 'topic': 'Reinforcement Learning'}, page_content='in RL include Q-learning, Deep Q-Networks (DQN), and Proximal Policy Optimization (PPO).\n\nRL has been successfully applied in various domains, including robotics, game playing (e.g., AlphaGo), and autonomous systems.')]

In [28]:
# Add new chunks to existing ChromaDB vector store
vectorstore.add_documents(new_chunks)
vectorstore

In [29]:
print(f"After adding new document, vector store now has {vectorstore._collection.count()} vectors.")

After adding new document, vector store now has 14 vectors.


In [30]:
# Query with the updated vector store

query = "What are key concepts in Reinforcement Learning?"
print(f"Question: {query}\n")
results = rag_chain_lcel.invoke(query)
print(f"Answer: {results}\n")
print("-" * 50)


Question: What are key concepts in Reinforcement Learning?

===== Retrieved Documents inside chain =====
Doc 1: Reinforcement Learning (RL)

Reinforcement Learning (RL) is a type of machine learning where an agent learns to make decisions by
interacting with an environment. The agent receives feedback in the form of rewards or penalties based on its actions, 
and its goal is to learn a policy that maximizes cumulative rewards over time. 

Key Concepts in RL: states, actions, rewards, policy, value function, and exploration vs. exploitation.
Popular RL algorithms in RL include Q-learning, Deep Q-Networks

Doc 2: in RL include Q-learning, Deep Q-Networks (DQN), and Proximal Policy Optimization (PPO).

RL has been successfully applied in various domains, including robotics, game playing (e.g., AlphaGo), and autonomous systems.

Doc 3: learning learns through
    interactions with an environment using rewards and penalties.

Answer: Key concepts in Reinforcement Learning include states, ac

### Advance RAG Techniques - Conversational Memory

In [31]:
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough, RunnableSequence
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter

# Step 1: Prompt
contextualize_a_system_prompt = """Given a chat history and the latest user question
which might reference context in the chat history, formulate a standalone question
which can be understood without the chat history. DO NOT answer the question, just
reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_a_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{latest_question}")
])

# Step 2: Reformulate question
reformulate_question = (
    {
        "chat_history": itemgetter("chat_history"),
        "latest_question": itemgetter("latest_question"),
    }
    | contextualize_q_prompt
    | llm
    | StrOutputParser()
)
reformulate_question

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer ONLY from context. If unknown, say you don't know."),
    MessagesPlaceholder("chat_history"),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

# -----------------------------
# 8. FINAL RAG CHAIN (LCEL)
# -----------------------------
rag_chain = (
    {
        # reformulated question
        "question": reformulate_question,
        "chat_history": itemgetter("chat_history"),
    }
    | {
        "question": itemgetter("question"),
        "chat_history": itemgetter("chat_history"),
        "context": itemgetter("question") | retriever | format_docs,
    }
    | qa_prompt
    | llm
    | StrOutputParser()
)

# -----------------------------
# 9. Test
# -----------------------------
chat_history = []
   

response = rag_chain.invoke({
    "question": "what is machine learning?",
    "chat_history": chat_history
})

print("\nAnswer:\n")
print(response)


Answer:

Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.


In [32]:
chat_history

[]

In [33]:
chat_history.extend([
    HumanMessage(content="what is machine learning?"),
    AIMessage(content=response)
])

In [34]:
# follow-up question that references previous question without restating it
follow_up_response = rag_chain.invoke({
    "question": "what are its main types?",
    "chat_history": chat_history
})
follow_up_response

'The main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning.'

In [35]:
chat_history.extend([
    HumanMessage(content="what are its main types?"),
    AIMessage(content=follow_up_response)
])
chat_history

[HumanMessage(content='what is machine learning?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='what are its main types?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [36]:
follow_up_response = rag_chain.invoke({
    "question": "Uses of Supervised Learning?",
    "chat_history": chat_history
})
follow_up_response

'Supervised learning uses labeled data to train models.'

In [37]:
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough, RunnableSequence
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter

# Step 1: Prompt
contextualize_a_system_prompt = """Given a chat history and the latest user question
which might reference context in the chat history, formulate a standalone question
which can be understood without the chat history. DO NOT answer the question, just
reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_a_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{latest_question}")
])

# Step 2: Reformulate question
reformulate_question = (
    {
        "chat_history": itemgetter("chat_history"),
        "latest_question": itemgetter("latest_question"),
    }
    | contextualize_q_prompt
    | llm
    | StrOutputParser()
)
reformulate_question

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer ONLY from context. If unknown, say you don't know."),
    MessagesPlaceholder("chat_history"),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

# -----------------------------
# 8. FINAL RAG CHAIN (LCEL)
# -----------------------------
rag_chain = (
    {
        # reformulated question
        "question": reformulate_question,
        "chat_history": itemgetter("chat_history"),
    }
    | {
        "question": itemgetter("question"),
        "chat_history": itemgetter("chat_history"),
        "context": itemgetter("question") | retriever | format_docs,
    }
    | qa_prompt
    | llm
    | StrOutputParser()
)

# -----------------------------
# 9. Test
# -----------------------------
def ask_question(chain, question, chat_history):
    response = chain.invoke({
        "question": question,
        "chat_history": chat_history
    })
    
    chat_history.extend([
        HumanMessage(content=question),
        AIMessage(content=response)
    ])
    
    return response

chat_history = []
print(chat_history)
res1 = ask_question(rag_chain, "what is machine learning?", chat_history)
print(f"Question: What is machine learning?\n")
print("\nAnswer:\n")
print(response)

print(chat_history)
res2 = ask_question(rag_chain, "what are its main types?", chat_history)
print(f"Question: What are its main types?\n")
print("\nAnswer:\n")
print(res2)

print(chat_history)
res3 = ask_question(rag_chain, "Uses of Supervised Learning?", chat_history)
print(f"Question: Uses of Supervised Learning?\n")
print("\nAnswer:\n")
print(res3)

[]
Question: What is machine learning?


Answer:

Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.
[HumanMessage(content='what is machine learning?', additional_kwargs={}, response_metadata={}), AIMessage(content='Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
Question: What are its main types?


Answer:

The main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning.
[HumanMessage(content='what is machine learning?', additional_kwargs={}, response_metadata={}), AIMessage(content='Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.', additional_kwargs={}, r

In [38]:
from operator import itemgetter

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_text_splitters import RecursiveCharacterTextSplitter


# -----------------------------
# 1. LLM
# -----------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# -----------------------------
# 2. Documents
# -----------------------------
docs = [
    Document(page_content="Tesla is an American electric vehicle company."),
    Document(page_content="The CEO of Tesla is Elon Musk."),
    Document(page_content="Tesla was founded in 2003."),
]

# -----------------------------
# 3. Split
# -----------------------------
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = splitter.split_documents(docs)

# -----------------------------
# 4. Embeddings + Chroma
# -----------------------------
embeddings = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# -----------------------------
# 5. Prompt: Reformulate Question
# -----------------------------
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the question to be standalone. Do NOT answer."),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}")
])

# Reformulation chain
reformulate_chain = (
    {
        "chat_history": itemgetter("chat_history"),
        "question": itemgetter("question"),
    }
    | contextualize_prompt
    | llm
    | StrOutputParser()
)

# -----------------------------
# 6. Retriever Function
# -----------------------------
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# -----------------------------
# 7. QA Prompt
# -----------------------------
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer ONLY from context. If unknown, say you don't know."),
    MessagesPlaceholder("chat_history"),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

# -----------------------------
# 8. FINAL RAG CHAIN (LCEL)
# -----------------------------
rag_chain = (
    {
        # reformulated question
        "question": reformulate_chain,
        "chat_history": itemgetter("chat_history"),
    }
    | {
        "question": itemgetter("question"),
        "chat_history": itemgetter("chat_history"),
        "context": itemgetter("question") | retriever | format_docs,
    }
    | qa_prompt
    | llm
    | StrOutputParser()
)

# -----------------------------
# 9. Test
# -----------------------------
chat_history = [
    HumanMessage(content="Tell me about Tesla"),
    AIMessage(content="Tesla is an electric vehicle company.")
]

response = rag_chain.invoke({
    "question": "Who is its CEO?",
    "chat_history": chat_history
})

print("\nAnswer:\n")
print(response)


Answer:

The CEO of Tesla is Elon Musk.


In [39]:
llm

ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001CC2D15B790>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001CC2D15BAD0>, root_client=<openai.OpenAI object at 0x000001CC2D159AD0>, root_async_client=<openai.AsyncOpenAI object at 0x000001CC2D159F50>, model_name='gpt-4o-mini', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)